In [147]:
%run classes/beautify_page.ipynb
%run classes/my_pandas.ipynb
from selenium import webdriver
from selenium.webdriver.chrome.options import Options # headless request, without chrome window open
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
# import below class for regex
import re 
import pymysql
import matplotlib.pyplot as plt


In [46]:
class MyLib(BeautifyPage, MyPandas):
    
    def extract_price(self, price_text):
        """
        Added this method to extract any currency having different currency symbols attached.
        This function will extract numeric or decimal nums.
        """
        if not price_text:
            return None
        # Extract numbers and decimals
        match = re.search(r"[\d,.]+", price_text)
        if not match:
            return None
        # Remove commas and convert to float
        try:
            return float(match.group(0).replace(",", ""))
        except ValueError:
            return None


# Initialize driver (once). It is used to scrap data using sellinium in new window
driver = webdriver.Chrome()
driver.maximize_window()

In [121]:
import pymysql

def create_connection():
    """Create and return database connection"""
    connection = pymysql.connect(
        host='localhost',
        user='root',
        password='',
        database='banggood',
        charset='utf8mb4',
        cursorclass=pymysql.cursors.DictCursor
    )
    return connection


def insert_categories(cursor, categories_data):
    """
    Insert categories into the categories table and return a mapping of category names to category_ids.
    :return: A dictionary mapping category names to cat_ids.
    """
    category_to_id_map = {}

    for category in categories_data:
        cursor.execute('''
            INSERT INTO categories (name) 
            VALUES (%s)
        ''', (category['name'],))
        cursor.execute('SELECT LAST_INSERT_ID()')
        cat_id = cursor.fetchone()['LAST_INSERT_ID()']
        category_to_id_map[category['name']] = cat_id
    
    return category_to_id_map


def insert_category_items(cursor, category_items_data, category_to_id_map):
    """
    Insert product items into the category_items table using the cat_id from the category_to_id_map.
    :param cursor: The cursor object to interact with the database
    :param category_items_data: A list of dictionaries containing product data
    :param category_to_id_map: A dictionary mapping category names to cat_ids
    """
    for item in category_items_data:
        cat_id = category_to_id_map.get(item['category'])

        if not cat_id:
            print(f"Category {item['category']} not found in categories table.")
            continue

        cursor.execute('''
            INSERT INTO category_items 
            (category_id, title, total_reviews, total_stars, discount_price, old_price, url) 
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        ''', (
            cat_id, 
            item['title'],
            item['total_reviews'],
            item['total_stars'],
            item['discount_price'],
            item['old_price'],
            item['url']
        ))
        print(f"Inserted product: {item['title']} into category {item['category']}")

In [41]:
my_lib = MyLib()
cat_one_url = 'http://banggood.com/Deals_Cell_Phones_Accessories.html?rmmds=PhonesFlashDeals&bid=80821&selectedPid=2035160'
# Iterable of 5 category URLs
category_urls = {
    "cell_phone_accessories": "http://banggood.com/Deals_Cell_Phones_Accessories.html?rmmds=PhonesFlashDeals&bid=80821&selectedPid=2035160",
    "computer_videos_and_games": "https://www.banggood.com/Wholesale-Monitors-Accessories-ca-5015.html?bid=80825&from=nav",
    "automobile_and_motorcycle": "https://www.banggood.com/Wholesale-Car-Heater-ca-4133.html?bid=80810",
    "home_and_furniture": "https://www.banggood.com/Wholesale-Watering-and-Irrigation-ca-12210.html?cat_id=12210&page=1&direct=0&rec_uid=2693119880|1763832121&bid=80830&sort=1&sortType=desc",
    "clothing": "https://www.banggood.com/Wholesale-T-Shirts-ca-15003.html?bid=81127&from=nav"
}

In [47]:
# Lists to store data for insertion
categories_data = []
category_items_data = []

# Loop through categories
for category, cat_url in category_urls.items():
    driver.get(cat_url)
     #wait until script is loaded
    WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.fg-listWrap"))
    )
    
    # Parse category page
    soup = BeautifulSoup(driver.page_source, "html.parser")
    response = soup.select_one("div.fg-list")
    
    if not response:
        print(f"No products found for category: {category}")
        continue

    # Add category to categories_data if not already added
    if category not in [cat['name'] for cat in categories_data]:
        categories_data.append({"name": category})

    # Loop through product links (max 5 items)
    for idx, a_tag in enumerate(response.find_all('a', class_='img-wrap')):
        href = a_tag.get('href')
        if not href:
            continue
        driver.get(href) # selenium for script rendering
        try:
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "span.main-price"))
            )
        except:
            print(f"Price not found for: {href}")
            continue
        
        # Parse product page
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        product_title = soup.select_one("span.product-title-text")
        product_total_review = soup.select_one("span.reviews-num")
        product_total_stars = soup.select_one("span.star-num")
        new_price_tag = soup.select_one("span.main-price")
        old_price_tag = soup.select_one("span.old-price")

        old_price = old_price_tag.text.strip().replace("$", "").replace(",", "") if old_price_tag else None
        new_price = new_price_tag.text.strip().replace("$", "").replace(",", "") if new_price_tag else None
        
        # Convert to float if values exist
        try:
            old_price = my_lib.extract_price(old_price)
            new_price = my_lib.extract_price(new_price)
        except ValueError:
            old_price = None
            new_price = None
        
        # Now calculate derived values safely
        price_diff = old_price - new_price if old_price and new_price else None
        discount_percent = (price_diff / old_price) * 100 if price_diff and old_price else None
        
        data = {
            "category": category,
            "url": href,
            "title": product_title.get('data-text') if product_title else None,
            "total_reviews": product_total_review.text.strip() if product_total_review else None,
            "total_stars": product_total_stars.text.strip() if product_total_stars else None,
            "discount_price": new_price_tag if new_price_tag else None,
            "old_price": old_price_tag if old_price_tag else None,
            ### Two additional derived values
            "price_diff": price_diff,
            "discount_percent": discount_percent,
        }
        my_lib.add_row(data)

        # Prepare data for category_items table
        item_data = {
            "category": category,  # Will be linked to category using category name
            "url": href,
            "title": product_title.get('data-text') if product_title else None,
            "total_reviews": product_total_review.text.strip() if product_total_review else None,
            "total_stars": product_total_stars.text.strip() if product_total_stars else None,
            "discount_price": new_price,
            "old_price": old_price,
            "price_diff": price_diff,
            "discount_percent": discount_percent,
        }

        # Add product data to category_items_data
        category_items_data.append(item_data)

# Convert all collected data to a DataFrame
df = my_lib.to_dataframe()
# Quit driver when done
driver.quit()

TimeoutException: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff73026a235
	0x7ff72ffc2630
	0x7ff72fd516dd
	0x7ff72fdaa27e
	0x7ff72fdaa58c
	0x7ff72fdfed77
	0x7ff72fdfbaba
	0x7ff72fd9b0ed
	0x7ff72fd9bf63
	0x7ff730295d60
	0x7ff73028fe8a
	0x7ff7302b1005
	0x7ff72ffdd71e
	0x7ff72ffe4e1f
	0x7ff72ffcb7c4
	0x7ff72ffcb97f
	0x7ff72ffb18e8
	0x7fffea69e8d7
	0x7fffeb30c53c


# ---------------------------------
### 3. Python Exploratory Analysis [Dummy data for testing]
# ---------------------------------
dummy_categories =  [
    {'name': 'cell_phone_accessories'},
    {'name': 'clothing'},
    {'name': 'Shome_and_furniture'},
    {'name': 'automobile_and_motorcycle'},
    {'name': 'computer_videos_and_games'},
]
dummy_data = [
    # ------------------------ Category 1 ------------------------
    {
        "category": "cell_phone_accessories",
        "url": "http://example.com/p1",
        "title": "Wireless Earbuds Pro",
        "total_reviews": "1245",
        "total_stars": "4.7",
        "discount_price": 29.99,
        "old_price": 49.99,
        "price_diff": 49.99 - 29.99,
        "discount_percent": ((49.99 - 29.99) / 49.99) * 100,
    },
    {
        "category": "cell_phone_accessories",
        "url": "http://example.com/p2",
        "title": "Fast Charging Power Bank 20000mAh",
        "total_reviews": "857",
        "total_stars": "4.5",
        "discount_price": 21.99,
        "old_price": 34.99,
        "price_diff": 34.99 - 21.99,
        "discount_percent": ((34.99 - 21.99) / 34.99) * 100,
    },
    {
        "category": "cell_phone_accessories",
        "url": "http://example.com/p3",
        "title": "Magnetic Phone Holder",
        "total_reviews": "645",
        "total_stars": "4.4",
        "discount_price": 8.99,
        "old_price": 14.99,
        "price_diff": 14.99 - 8.99,
        "discount_percent": ((14.99 - 8.99) / 14.99) * 100,
    },
    {
        "category": "cell_phone_accessories",
        "url": "http://example.com/p4",
        "title": "USB-C Fast Charger 30W",
        "total_reviews": "234",
        "total_stars": "4.6",
        "discount_price": 12.49,
        "old_price": 19.99,
        "price_diff": 19.99 - 12.49,
        "discount_percent": ((19.99 - 12.49) / 19.99) * 100,
    },
    {
        "category": "cell_phone_accessories",
        "url": "http://example.com/p5",
        "title": "Premium Tempered Glass",
        "total_reviews": "1200",
        "total_stars": "4.8",
        "discount_price": 4.99,
        "old_price": 9.99,
        "price_diff": 9.99 - 4.99,
        "discount_percent": ((9.99 - 4.99) / 9.99) * 100,
    },

    # ------------------------ Category 2 ------------------------
    {
        "category": "computer_videos_and_games",
        "url": "http://example.com/p6",
        "title": "Gaming Mouse RGB",
        "total_reviews": "1890",
        "total_stars": "4.6",
        "discount_price": 24.99,
        "old_price": 39.99,
        "price_diff": 39.99 - 24.99,
        "discount_percent": ((39.99 - 24.99) / 39.99) * 100,
    },
    {
        "category": "computer_videos_and_games",
        "url": "http://example.com/p7",
        "title": "Mechanical Keyboard Blue Switch",
        "total_reviews": "2200",
        "total_stars": "4.7",
        "discount_price": 44.99,
        "old_price": 59.99,
        "price_diff": 59.99 - 44.99,
        "discount_percent": ((59.99 - 44.99) / 59.99) * 100,
    },
    {
        "category": "computer_videos_and_games",
        "url": "http://example.com/p8",
        "title": "1080p Webcam",
        "total_reviews": "990",
        "total_stars": "4.3",
        "discount_price": 19.99,
        "old_price": 34.99,
        "price_diff": 34.99 - 19.99,
        "discount_percent": ((34.99 - 19.99) / 34.99) * 100,
    },
    {
        "category": "computer_videos_and_games",
        "url": "http://example.com/p9",
        "title": "Gaming Headset 7.1",
        "total_reviews": "3100",
        "total_stars": "4.5",
        "discount_price": 29.99,
        "old_price": 49.99,
        "price_diff": 49.99 - 29.99,
        "discount_percent": ((49.99 - 29.99) / 49.99) * 100,
    },
    {
        "category": "computer_videos_and_games",
        "url": "http://example.com/p10",
        "title": "4K HDMI Cable",
        "total_reviews": "580",
        "total_stars": "4.8",
        "discount_price": 6.99,
        "old_price": 12.99,
        "price_diff": 12.99 - 6.99,
        "discount_percent": ((12.99 - 6.99) / 12.99) * 100,
    },

    # ------------------------ Category 3 ------------------------
    {
        "category": "automobile_and_motorcycle",
        "url": "http://example.com/p11",
        "title": "Car Phone Mount",
        "total_reviews": "1450",
        "total_stars": "4.7",
        "discount_price": 11.99,
        "old_price": 19.99,
        "price_diff": 19.99 - 11.99,
        "discount_percent": ((19.99 - 11.99) / 19.99) * 100,
    },
    {
        "category": "automobile_and_motorcycle",
        "url": "http://example.com/p12",
        "title": "Motorcycle LED Light",
        "total_reviews": "600",
        "total_stars": "4.3",
        "discount_price": 14.99,
        "old_price": 24.99,
        "price_diff": 24.99 - 14.99,
        "discount_percent": ((24.99 - 14.99) / 24.99) * 100,
    },
    {
        "category": "automobile_and_motorcycle",
        "url": "http://example.com/p13",
        "title": "Car Vacuum Cleaner",
        "total_reviews": "2200",
        "total_stars": "4.1",
        "discount_price": 25.99,
        "old_price": 39.99,
        "price_diff": 39.99 - 25.99,
        "discount_percent": ((39.99 - 25.99) / 39.99) * 100,
    },

    # ------------------------ Category 4 ------------------------
    {
        "category": "home_and_furniture",
        "url": "http://example.com/p14",
        "title": "LED Bedside Lamp",
        "total_reviews": "450",
        "total_stars": "4.6",
        "discount_price": 18.99,
        "old_price": 29.99,
        "price_diff": 29.99 - 18.99,
        "discount_percent": ((29.99 - 18.99) / 29.99) * 100,
    },
    {
        "category": "home_and_furniture",
        "url": "http://example.com/p15",
        "title": "Smart Aroma Diffuser",
        "total_reviews": "1300",
        "total_stars": "4.4",
        "discount_price": 22.99,
        "old_price": 39.99,
        "price_diff": 39.99 - 22.99,
        "discount_percent": ((39.99 - 22.99) / 39.99) * 100,
    },

    # ------------------------ Category 5 ------------------------
    {
        "category": "clothing",
        "url": "http://example.com/p16",
        "title": "Men's Casual Shirt",
        "total_reviews": "780",
        "total_stars": "4.2",
        "discount_price": 14.99,
        "old_price": 24.99,
        "price_diff": 24.99 - 14.99,
        "discount_percent": ((24.99 - 14.99) / 24.99) * 100,
    },
    {
        "category": "clothing",
        "url": "http://example.com/p17",
        "title": "Women's Summer Dress",
        "total_reviews": "1500",
        "total_stars": "4.8",
        "discount_price": 19.99,
        "old_price": 34.99,
        "price_diff": 34.99 - 19.99,
        "discount_percent": ((34.99 - 19.99) / 34.99) * 100,
    },
]

df = pd.DataFrame(dummy_data)

In [54]:
## 1. Category-Wise Average Discount Percentage
df.groupby("category")["discount_percent"].mean()

category
automobile_and_motorcycle    38.348256
cell_phone_accessories       40.951394
clothing                     41.442699
computer_videos_and_games    38.316063
home_and_furniture           39.594760
Name: discount_percent, dtype: float64

In [59]:
## 2. Correlation Between Discount and Popularity
df[["discount_percent", "total_reviews", "total_stars"]].corr()

,discount_percent,total_reviews,total_stars
discount_percent,1.000000,-0.297880,0.178826
total_reviews,-0.297880,1.000000,-0.011539
total_stars,0.178826,-0.011539,1.000000


In [61]:
## 3.Top 10 highest discounted products
df.nlargest(10, 'discount_percent').sort_values('category')[['title', 'category', 'old_price', 'discount_price', 'discount_percent']]

,title,category,old_price,discount_price,discount_percent
10,Car Phone Mount,automobile_and_motorcycle,19.99,11.99,40.020010
11,Motorcycle LED Light,automobile_and_motorcycle,24.99,14.99,40.016006
4,Premium Tempered Glass,cell_phone_accessories,9.99,4.99,50.050050
2,Magnetic Phone Holder,cell_phone_accessories,14.99,8.99,40.026684
0,Wireless Earbuds Pro,cell_phone_accessories,49.99,29.99,40.008002
16,Women's Summer Dress,clothing,34.99,19.99,42.869391
15,Men's Casual Shirt,clothing,24.99,14.99,40.016006
9,4K HDMI Cable,computer_videos_and_games,12.99,6.99,46.189376
7,1080p Webcam,computer_videos_and_games,34.99,19.99,42.869391
14,Smart Aroma Diffuser,home_and_furniture,39.99,22.99,42.510628


In [66]:
## 4. Category wise price bracket
## This methodology describes how many products lies in the price bracked for each category. 
### For example there are 2 clothing products lies in range 0-30 ###
bins = [0, 30, 60, 90, 120, float('inf')]
labels = ['0-30', '30-60', '60-90', '90-120', '120+']

df['price_bracket'] = pd.cut(df['discount_price'], bins=bins, labels=labels)
df.groupby(['category', 'price_bracket'], observed=False).size().unstack(fill_value=0)

price_bracket,0-30,30-60,60-90,90-120,120+
category,,,,,
automobile_and_motorcycle,3,0,0,0,0
cell_phone_accessories,5,0,0,0,0
clothing,2,0,0,0,0
computer_videos_and_games,4,1,0,0,0
home_and_furniture,2,0,0,0,0


In [ ]:
# ----------------- #
# CREATE DB & Tables, if not exists
# ----------------- #
connection = pymysql.connect(
    host='localhost',
    user='root',
    password='',
    #database='unify_shan',
    charset='utf8mb4',
    cursorclass=pymysql.cursors.DictCursor
)

db_ops = DBOps()
db_ops.main
with connection:
    with connection.cursor() as cursor:
        cursor.execute('CREATE DATABASE IF NOT EXISTS banggood')
        print("Database 'banggood' is created or already exists.")
        
        cursor.execute('USE banggood')
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS categories (
                id INT AUTO_INCREMENT PRIMARY KEY,
                name VARCHAR(255) NOT NULL
            )
        ''')

        # Create category_items table
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS category_items (
                id INT AUTO_INCREMENT PRIMARY KEY,
                category_id INT,
                title VARCHAR(255) NOT NULL,
                total_reviews INT DEFAULT 0,
                total_stars FLOAT DEFAULT 0.0,
                discount_price DECIMAL(10, 2) DEFAULT 0.0,
                old_price DECIMAL(10, 2) DEFAULT 0.0,
                url VARCHAR(255) NOT NULL,
                FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE CASCADE
            )
        ''')
        print("Tables created or already exists in the system.")
        

In [ ]:
# Create connection
connection = create_connection()

try:
    with connection.cursor() as cursor:
        # Insert categories and get the mapping
        category_to_id_map = insert_categories(cursor, dummy_categories)
        
        # Insert category items
        insert_category_items(cursor, dummy_data, category_to_id_map)
        
        # Commit changes
        connection.commit()
    
    print("All data inserted successfully!")
    
except Exception as e:
    connection.rollback()
    print(f"Error: {e}")
    
finally:
    connection.close()

In [152]:
## 1. Get category-wise analytics
connection = create_connection()
with connection.cursor() as cursor:
    # 1. Category Statistics
    cursor.execute('''
        SELECT 
            c.name AS category_name,
            COUNT(ci.id) AS total_products,
            AVG(ci.total_stars) AS avg_rating,
            SUM(ci.total_reviews) AS total_reviews,
            AVG(ci.discount_price) AS avg_price,
            AVG((ci.old_price - ci.discount_price) / ci.old_price * 100) AS avg_discount_percent
        FROM categories c
        LEFT JOIN category_items ci ON c.id = ci.category_id
        GROUP BY c.id, c.name
        ORDER BY total_products DESC
    ''')
    
    stats = cursor.fetchall()  # Fetch the results here
    for row in stats:
        print(f"{row['category_name']}: {row['total_products']} products, "
              f"Avg Rating: {row['avg_rating']}, "
              f"Avg Discount: {row['avg_discount_percent']}%")